In [6]:
# Dummy data set 
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio as rio
import shutil
import zipfile
import os
import io
from shapely.geometry import box
import time
import urllib.request
from urllib.parse import urlparse

In [12]:
# Excel reader
!pip install openpyxl

# Load the dataset from Zenodo
zenodo_url = "https://zenodo.org/records/20118715/files/MPs_WWTPs_Eff_DUMMY.xlsx?download=1"

# Define raw data folder in diwa 
raw_data_path = Path.home()/"data"/"raw"
raw_data_path.mkdir(parents=True, exist_ok=True)

# Output destination
download_path = raw_data_path / "MPs_WWTPs_Eff_DUMMY.xlsx"

# Download file
response = requests.get(zenodo_url)
response.raise_for_status()

with open(download_path, "wb") as f:
    f.write(response.content)

print("Downloaded dataset saved to:", download_path)

# Read the Excel file
data = pd.read_excel(download_path)

data.head()

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 6.6 MB/s eta 0:00:00:00:01
Downloaded dataset saved to: data/raw/MPs_WWTPs_Eff_DUMMY.xlsx


,WWTP / Env.,Municipality,Date,Sampling Point,Project,Monitoring Interval,Season,WWTP,Tech type,PE,...,Sucralose,Sulfapyridine,Telmisartan,Tramadol,Trimetoprim,Valsartan,Valsartan acid,Venlafaxine,Venlafaxine O-desmethyl,Total_Concentration_ngL
0,Site 1,City 1,2024-10-04,CAS_S1_5,Project XXXX,Seasonally,Fall 2024,WWTP 1A,CAS,21480.46,...,45860.97,2452.33,2026.55,114.35,349.61,4770.29,77.81,655.26,1369.85,298065.0
1,Site 1,City 1,2024-10-30,MBR_S1_7,Project XXXX,Seasonally,Fall 2024,WWTP 1B,MBR,80391.15,...,54066.31,661.41,2089.36,312.34,269.15,12647.16,1283.40,512.79,1139.59,173127.7
2,Site 2,City 2,2024-11-05,Post_S2_5,Project XXXX,Seasonally,Fall 2024,WWTP 2,CAS,12607.66,...,48321.51,2198.68,1599.23,347.75,382.38,3290.60,1229.50,440.15,2203.82,247389.0
3,Site 3,City 3,2024-11-12,Post_S3_5,Project XXXX,Seasonally,Fall 2024,WWTP 3,CAS,29796.30,...,46125.70,141.70,1172.45,223.34,281.19,30553.73,1849.30,393.36,1813.38,386035.6
4,Site 4,City 3,2024-10-22,Post_S4_5,Project XXXX,Seasonally,Fall 2024,WWTP 4,MBBR,72894.36,...,71903.66,1492.18,1267.70,266.83,363.03,2232.00,412.85,508.30,2296.57,326693.3


In [17]:
# =========================
# API DATA ACCESS (OSM)
# =========================

import requests
import geopandas as gpd
from shapely.geometry import shape

# Overpass (OpenStreetMap) API endpoint (more stable mirror)
overpass_url = "https://overpass.kumi.systems/api/interpreter"

# STEP 1: Define the query (water features in Oulu area)
query = """
[out:json];
(
  way["natural"="water"](65.00,25.00,65.20,25.60);
);
out geom;
"""

# STEP 2: Send request to the API (with timeout)
response = requests.post(overpass_url, data={"data": query}, timeout=60)
response.raise_for_status()

data_osm = response.json()

# STEP 3: Convert API response to GeoDataFrame
geoms = []
for element in data_osm["elements"]:
    if "geometry" in element:
        geom = {
            "name": element.get("tags", {}).get("name", None),
            "geometry": shape({
                "type": "LineString",
                "coordinates": [(pt["lon"], pt["lat"]) for pt in element["geometry"]]
            })
        }
        geoms.append(geom)

gdf = gpd.GeoDataFrame(geoms)

# STEP 4: Save API results
api_output_path = "data/raw/osm_water_features.geojson"
gdf.to_file(api_output_path, driver="GeoJSON")

gdf.head()

/home/jovyan/.local/lib/python3.11/site-packages/pyogrio/geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


,name,geometry
0,Kalikkalampi,"LINESTRING (25.55602 65.067, 25.55594 65.06713..."
1,Hämeenjärvi,"LINESTRING (25.5776 65.14024, 25.57813 65.1399..."
2,Kummunlampi,"LINESTRING (25.59686 65.13213, 25.5974 65.1317..."
3,Takalampi,"LINESTRING (25.59351 65.12807, 25.59482 65.127..."
4,Martinlampi,"LINESTRING (25.59786 65.20129, 25.59886 65.201..."
